In [6]:
"""
Greedy algorithms typically have this top-down design:
make a choise and then solve a subproblem, rather than
the bottom-up technique of solving subproblems before making
a choice.

stores the indexes of the
start time of candidate ≥ finish time of last selected

activity_selector example:
s = [0, 1, 3, 0, 5, 3, 5, 6, 8, 8, 2, 12]
f = [0, 4, 5, 6, 7, 9, 9, 10, 11, 12, 14, 16]
[1] + [4] + [8] + [11] + []
1 > 0, 5 > 4, 8 > 7, 12 > 11

simply picks times that don't overlap
"""
def recursive_activity_selector(s, f, k):
    m = k + 1
    
    while m < len(s) and s[m] < f[k]:
        m += 1
    
    if m < len(s):
        return [m] + recursive_activity_selector(s, f, m)
    else:
        return []

In [4]:
def greedy_activity_selector(s, f):
    selected = [0]
    k = 0
    
    for m in range(1, len(s)):
        if s[m] >= f[k]:
            selected.append(m)
            k = m
            
    return selected

In [5]:
def greedy_activity_selector_clean(activities):
    activities = sorted(activities, key=lambda x: x[1])
    
    selected = [activities[0]]
    last_finish = activities[0][1]
    
    for start, finish in activities[1:]:
        if start >= last_finish:
            selected.append((start, finish))
            last_finish = finish
            
    return selected

In [7]:
"""
The DP solution proves correctness structure:
Optimal solution = optimal left + optimal right + chosen activity

The greedy solution exploits a structural property:
Choosing the activity that finishes earliest always leaves maximum room for the rest.

That property collapses O(n³) → O(n).
The maximum number of mutually compatible activities we can select
c[i][j] = maximum number of non-overlapping activities
          that fit strictly between activity i and activity j

Imagine this:
i -------------------- j

We’re trying to pack activities between those boundaries.
For each possible choice k:
i ---- k ---- j

We split into two independent subproblems:
Left side: (i, k)
Right side: (k, j)
Then combine:
best_left + best_right + 1

So the DP table stores:
The size of the best schedule in every possible sub-interval.
"""
def activity_selector_dp(s, f):
    n = len(s)
    
    c = [[0] * n for _ in range(n)]
    solution = [[None] * n for _ in range(n)]
    
    # gap = distance between i and j
    for gap in range(2, n):
        for i in range(n - gap):
            j = i + gap
            
            for k in range(i + 1, j):
                if f[i] <= s[k] and f[k] <= s[j]:
                    value = c[i][k] + c[k][j] + 1
                    
                    if value > c[i][j]:
                        c[i][j] = value
                        solution[i][j] = k
    
    return c, solution

In [8]:
def reverse_greedy_activity_selector(s, f):
    n = len(s)
    selected = [n - 1]   # pick activity that starts last
    k = n - 1
    
    for m in range(n - 2, -1, -1):
        if f[m] <= s[k]:
            selected.append(m)
            k = m
            
    return selected[::-1]

In [9]:
"""
Minimum lecture halls needed: 3

Assignments:
Activity (1, 4) -> Hall 1
Activity (2, 5) -> Hall 2
Activity (3, 6) -> Hall 3
Activity (4, 7) -> Hall 1

Activities (sorted by start time):
A (1,4)
B (2,5)
C (3,6)
D (4,7)
------------------------------------

Activity A (1,4)
Heap empty → open Hall 1.
Heap:
[(4, H1)]

------------------------------------

Activity B (2,5)
Earliest finish = 4
4 > 2 → Hall 1 not free.
Open Hall 2.
Heap:
[(4, H1), (5, H2)]

------------------------------------

Activity C (3,6)
Earliest finish = 4
4 > 3 → no hall free.
Open Hall 3.
Heap:
[(4, H1), (5, H2), (6, H3)]

------------------------------------

Activity D (4,7)
Earliest finish = 4
4 <= 4 → Hall 1 becomes free.
Reuse Hall 1.
Remove (4, H1)
Insert (7, H1)
Heap:
[(5, H2), (6, H3), (7, H1)]

------------------------------------

Hall 1: A (1,4), D (4,7)
Hall 2: B (2,5)
Hall 3: C (3,6)

"""
import heapq

def interval_partitioning(activities):
    # activities: list of (start, finish)
    
    # Step 1: sort by start time
    activities.sort(key=lambda x: x[0])
    
    heap = []  # min-heap of (finish_time, hall_number)
    hall_count = 0
    assignment = []
    
    for start, finish in activities:
        # If there's a hall that becomes free
        if heap and heap[0][0] <= start:
            free_finish, hall = heapq.heappop(heap)
        else:
            hall_count += 1
            hall = hall_count
        
        assignment.append((start, finish, hall))
        heapq.heappush(heap, (finish, hall))
    
    return hall_count, assignment

In [10]:
"""
activities: list of (start, finish, value)
"""
import bisect

def weighted_activity_selection(activities):
    # Step 1: sort by finish time
    activities = sorted(activities, key=lambda x: x[1])
    
    n = len(activities)
    
    starts = [a[0] for a in activities]
    finishes = [a[1] for a in activities]
    values = [a[2] for a in activities]
    
    # Step 2: compute p array
    p = [0] * n
    for j in range(n):
        # find rightmost activity that finishes <= starts[j]
        i = bisect.bisect_right(finishes, starts[j]) - 1
        p[j] = i
    
    # Step 3: DP
    OPT = [0] * (n + 1)
    
    for j in range(1, n + 1):
        include = values[j - 1] + (OPT[p[j - 1] + 1] if p[j - 1] != -1 else 0)
        exclude = OPT[j - 1]
        OPT[j] = max(include, exclude)
    
    return OPT[n]

In [ ]:
"""
Greedy works all the time if the problem has 2 properties:
1) greedy-choice property
2) optimal substructure
"""